# PDF Scan v2 - Section-first rebuild

This notebook currently implements **Phase A** and **Phase B** from `PDF_SCAN_PIPELINE_IMPLEMENTATION_PLAN.md`.

- **Phase A**: config surface, run artifacts under `runs/{run_id}/`, structured logging, per-stage metrics, and a PDF manifest.
- **Phase B**: parser bundle creation with `PyMuPDF`, `pypdf`, optional `Docling`, optional `GROBID`, plus raw per-document diagnostics.
- Later phases will add canonical section construction, query planning, retrieval, reranking, and calibration.

**Ground rules for this version**

- Prefer loud, detailed errors over silent failure.
- Each major code cell ends with a compact QC summary.
- Ranking does not happen yet in this notebook version.


## Step 0 - Edit your inputs

Edit the next cell, then run the notebook top-to-bottom.

What the notebook should produce at this point:

- a stable `run_id`
- `runs/<run_id>/config.json`
- `runs/<run_id>/pdf_manifest.json`
- `runs/<run_id>/logs.jsonl`
- `runs/<run_id>/metrics.json`
- placeholder directories for later phases
- per-document parser bundles under `runs/<run_id>/parser/<doc_id>/`


In [1]:
# -----------------------------
# USER INPUTS (edit this cell)
# -----------------------------

import json
import re
from pathlib import Path

INPUT_MODE = "small_gold"  # "small_gold" or "manual"

# Benchmark mode: pull chapter + PDFs from the populated small-gold suite.
SMALL_GOLD_SUITE_MANIFEST = r"benchmark/small_gold/manifests/suite_manifest.json"
SMALL_GOLD_CHAPTER_INDEX = 0
SMALL_GOLD_DOC_LIMIT = None
SMALL_GOLD_INCLUDE_DOC_IDS = []
SMALL_GOLD_EXCLUDE_DOC_IDS = []

# Manual mode: keep this for later ad hoc runs outside the benchmark suite.
MANUAL_CHAPTER_TITLE = "Technische Grundlagen: Zero Trust Architecture (ZTA) in Unternehmensnetzwerken"

MANUAL_CHAPTER_DESCRIPTION = """
Ziel ist eine prazise, technische Fundierung von Zero Trust Architecture (ZTA) fur Unternehmens-IT (On-Prem, Cloud, Hybrid),
um spater eine konkrete ZTA-Einfuhrung bewerten und planen zu konnen. Dazu gehoren Begriffsdefinition, Kernprinzipien,
Referenzarchitekturen, Telemetrie, kontinuierliche Bewertung, Migration in Legacy-Umgebungen und messbare Bewertungskriterien.
Produktvergleiche, Buyer's Guides und rein allgemeine Kryptographie-Einfuhrungen gehoren nicht in den Scope.
""".strip()

# Manual mode, option A: explicit PDF list
PDF_SOURCES = [
    # {"label": "paper_1", "path": r"C:\\path\\to\\paper.pdf"},
]

# Manual mode, option B: discover PDFs from a directory when PDF_SOURCES is empty
PDF_DIR = r""
PDF_GLOB = "*.pdf"
PDF_RECURSIVE = False
MAX_PDFS = 20

PIPELINE_VERSION = "pdf_scan_v2"
FORCE_REBUILD_PHASE_A = False


def _fmt_int(x) -> str:
    try:
        return f"{int(x):,}"
    except Exception:
        return str(x)


def _truncate(text: str, max_len: int = 120) -> str:
    s = str(text or "")
    return s if len(s) <= max_len else (s[: max_len - 1] + "...")


def print_section(title: str, width: int = 80, char: str = "=") -> None:
    line = char * width
    print(line)
    print(title)
    print(line)


def print_kv(d: dict, key_width: int = 26) -> None:
    for k, v in d.items():
        print(f"{str(k):<{key_width}} {v}")


def _find_pdf_scan_dir_local() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for base in candidates:
        pdf_scan_dir = base / "pdf-scan"
        if pdf_scan_dir.exists() and pdf_scan_dir.is_dir():
            return pdf_scan_dir.resolve()
        if base.name == "pdf-scan" and base.is_dir():
            return base.resolve()
    raise RuntimeError("Could not resolve the pdf-scan directory from the current working directory.")


def _resolve_local_path(raw: str, *, expect_dir: bool) -> Path:
    p = Path(raw).expanduser()
    pdf_scan_dir = _find_pdf_scan_dir_local()
    repo_root = pdf_scan_dir.parent
    candidates = [p]
    if not p.is_absolute():
        candidates.extend([pdf_scan_dir / p, repo_root / p, Path.cwd().resolve() / p])
    seen = set()
    for cand in candidates:
        cand = cand.resolve()
        if cand in seen:
            continue
        seen.add(cand)
        if cand.exists() and ((cand.is_dir() and expect_dir) or (cand.is_file() and not expect_dir)):
            return cand
    return candidates[0].resolve()


def _load_json_local(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


BENCHMARK_SUITE_ID = ""
BENCHMARK_CHAPTER_ID = ""
BENCHMARK_SUITE_MANIFEST_RESOLVED = ""
BENCHMARK_SUITE_ROOT = ""

if INPUT_MODE == "small_gold":
    suite_manifest_path = _resolve_local_path(SMALL_GOLD_SUITE_MANIFEST, expect_dir=False)
    if not suite_manifest_path.exists():
        raise FileNotFoundError(f"Small-gold suite manifest not found: {suite_manifest_path}")

    suite_root = suite_manifest_path.parent.parent
    suite = _load_json_local(suite_manifest_path)
    chapter_specs = list(suite.get("chapter_specs") or [])
    if not chapter_specs:
        raise ValueError("The small-gold suite manifest contains no chapter_specs entries.")

    chapter_index = int(SMALL_GOLD_CHAPTER_INDEX)
    if chapter_index < 0 or chapter_index >= len(chapter_specs):
        raise IndexError(f"SMALL_GOLD_CHAPTER_INDEX out of range: {chapter_index}")

    chapter_path = (suite_root / str(chapter_specs[chapter_index])).resolve()
    chapter_spec = _load_json_local(chapter_path)

    include_doc_ids = {str(x).strip() for x in SMALL_GOLD_INCLUDE_DOC_IDS if str(x).strip()}
    exclude_doc_ids = {str(x).strip() for x in SMALL_GOLD_EXCLUDE_DOC_IDS if str(x).strip()}
    loaded_docs = []
    for rel_path in list(suite.get("documents") or []):
        doc_manifest_path = (suite_root / str(rel_path)).resolve()
        doc_manifest = _load_json_local(doc_manifest_path)
        doc_id = str(doc_manifest.get("doc_id") or "").strip()
        if include_doc_ids and doc_id not in include_doc_ids:
            continue
        if doc_id and doc_id in exclude_doc_ids:
            continue

        pdf_path = (suite_root / str(doc_manifest.get("path") or "")).resolve()
        if not pdf_path.exists():
            raise FileNotFoundError(f"Benchmark PDF not found: {pdf_path}")

        loaded_docs.append(
            {
                "label": str(doc_manifest.get("label") or pdf_path.stem).strip() or pdf_path.stem,
                "path": str(pdf_path),
                "doc_id": doc_id,
            }
        )

    if SMALL_GOLD_DOC_LIMIT is not None:
        loaded_docs = loaded_docs[: int(SMALL_GOLD_DOC_LIMIT)]
    if not loaded_docs:
        raise ValueError("Benchmark mode resolved zero PDFs after include/exclude filtering.")

    CHAPTER_TITLE = str(chapter_spec.get("title") or "").strip()
    CHAPTER_DESCRIPTION = str(chapter_spec.get("description") or "").strip()
    PDF_SOURCES = [{"label": row["label"], "path": row["path"]} for row in loaded_docs]
    PDF_DIR = r""
    PDF_GLOB = "*.pdf"
    PDF_RECURSIVE = False
    MAX_PDFS = len(PDF_SOURCES)

    BENCHMARK_SUITE_ID = str(suite.get("suite_id") or "").strip()
    BENCHMARK_CHAPTER_ID = str(chapter_spec.get("chapter_id") or "").strip()
    BENCHMARK_SUITE_MANIFEST_RESOLVED = str(suite_manifest_path)
    BENCHMARK_SUITE_ROOT = str(suite_root)
elif INPUT_MODE == "manual":
    CHAPTER_TITLE = MANUAL_CHAPTER_TITLE
    CHAPTER_DESCRIPTION = MANUAL_CHAPTER_DESCRIPTION
else:
    raise ValueError(f"Unsupported INPUT_MODE: {INPUT_MODE!r}")

if not str(CHAPTER_TITLE or "").strip():
    raise ValueError("CHAPTER_TITLE must not be empty.")
if not str(CHAPTER_DESCRIPTION or "").strip():
    raise ValueError("CHAPTER_DESCRIPTION must not be empty.")

desc_words = len(re.findall(r"\w+", CHAPTER_DESCRIPTION, flags=re.UNICODE))
source_mode = "SMALL_GOLD" if INPUT_MODE == "small_gold" else ("PDF_SOURCES" if PDF_SOURCES else ("PDF_DIR" if str(PDF_DIR or "").strip() else "unset"))

print_section("User Inputs")
print_kv(
    {
        "input_mode": INPUT_MODE,
        "chapter_title": _truncate(CHAPTER_TITLE, 90),
        "chapter_desc_chars": _fmt_int(len(CHAPTER_DESCRIPTION)),
        "chapter_desc_words": _fmt_int(desc_words),
        "pdf_source_mode": source_mode,
        "pipeline_version": PIPELINE_VERSION,
        "force_rebuild_phase_a": FORCE_REBUILD_PHASE_A,
        "benchmark_suite_id": BENCHMARK_SUITE_ID or "<none>",
        "benchmark_chapter_id": BENCHMARK_CHAPTER_ID or "<none>",
    }
)

print_section("User Inputs - PDF Discovery Config")
print_kv(
    {
        "pdf_sources_count": _fmt_int(len(PDF_SOURCES)),
        "pdf_dir": PDF_DIR or "<empty>",
        "pdf_glob": PDF_GLOB,
        "pdf_recursive": PDF_RECURSIVE,
        "max_pdfs": MAX_PDFS,
    }
)

if INPUT_MODE == "small_gold":
    print_section("User Inputs - Small Gold")
    print_kv(
        {
            "suite_manifest": _truncate(BENCHMARK_SUITE_MANIFEST_RESOLVED, 110),
            "suite_root": _truncate(BENCHMARK_SUITE_ROOT, 110),
            "chapter_index": SMALL_GOLD_CHAPTER_INDEX,
            "include_doc_ids": ", ".join(SMALL_GOLD_INCLUDE_DOC_IDS) if SMALL_GOLD_INCLUDE_DOC_IDS else "<all>",
            "exclude_doc_ids": ", ".join(SMALL_GOLD_EXCLUDE_DOC_IDS) if SMALL_GOLD_EXCLUDE_DOC_IDS else "<none>",
        }
    )

User Inputs
input_mode                 small_gold
chapter_title              Entscheidungspsychologie im Kontext unsicherer Kaufentscheidungen im Webshop-Kontext
chapter_desc_chars         665
chapter_desc_words         66
pdf_source_mode            SMALL_GOLD
pipeline_version           pdf_scan_v2
force_rebuild_phase_a      False
benchmark_suite_id         small_gold_webshop_decision_psychology_v1
benchmark_chapter_id       chapter_001_webshop_decision_psychology
User Inputs - PDF Discovery Config
pdf_sources_count          5
pdf_dir                    <empty>
pdf_glob                   *.pdf
pdf_recursive              False
max_pdfs                   5
User Inputs - Small Gold
suite_manifest             <projektverzeichnis>\pdf-scan\benchmark\small_gold\manifests\suite_manifest.json
suite_root                 <projektverzeichnis>\pdf-scan\benchmark\small_gold
chapter_index              0
include_doc_ids            <all>
exclude_doc_ids            <none>


---
# Phase A - Config, env loading, run artifacts, and structured logging
---

In [2]:
# Phase A.0 - Imports, repo resolution, env loading, and helper functions

import json
import logging
import os
import hashlib
import time
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional

from dotenv import load_dotenv

try:
    import fitz  # PyMuPDF
except Exception:
    fitz = None


def _find_repo_root_and_notebook_dir() -> tuple[Path, Path]:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for base in candidates:
        pdf_scan_dir = base / "pdf-scan"
        if pdf_scan_dir.exists() and pdf_scan_dir.is_dir():
            return base, pdf_scan_dir
        if base.name == "pdf-scan":
            return base.parent, base
    raise RuntimeError("Could not resolve repo root / pdf-scan directory from current working directory.")


REPO_ROOT, NOTEBOOK_DIR = _find_repo_root_and_notebook_dir()

load_dotenv(REPO_ROOT / ".env", override=False)
load_dotenv(NOTEBOOK_DIR / ".env", override=False)

OPENAI_API_KEY = (os.getenv("OPENAI_API_KEY") or "").strip()


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def fmt_float(x: Any, nd: int = 3) -> str:
    try:
        return f"{float(x):.{int(nd)}f}"
    except Exception:
        return str(x)


def fmt_ms(ms: Any) -> str:
    try:
        val = float(ms)
    except Exception:
        return str(ms)
    if val < 1000:
        return f"{val:.0f}ms"
    return f"{(val / 1000.0):.2f}s"


def print_table(rows: List[Dict[str, Any]], *, columns: List[str], max_rows: int = 50, max_col_width: int = 60) -> None:
    rows = list(rows or [])
    if not rows:
        print("<empty>")
        return

    show = rows[: int(max_rows)]
    cols = list(columns)

    def cell(row: Dict[str, Any], col: str) -> str:
        value = row.get(col, "")
        if value is None:
            value = ""
        text = str(value).replace("\r", " ").replace("\n", " ")
        return text if len(text) <= max_col_width else (text[: max_col_width - 3] + "...")

    widths = {}
    for col in cols:
        widths[col] = min(
            max(len(col), max(len(cell(r, col)) for r in show)),
            max_col_width,
        )

    header = " | ".join(f"{col:<{widths[col]}}" for col in cols)
    sep = "-+-".join("-" * widths[col] for col in cols)
    print(header)
    print(sep)
    for row in show:
        print(" | ".join(f"{cell(row, col):<{widths[col]}}" for col in cols))
    if len(rows) > len(show):
        print(f"... (+{len(rows) - len(show)} more rows)")


def qc_row(check: str, status: str, value: Any, expected: str, why: str, fix: str) -> Dict[str, Any]:
    return {
        "check": str(check),
        "status": str(status),
        "value": str(value),
        "expected": str(expected),
        "why": str(why),
        "fix": str(fix),
    }


def stable_hash(*parts: str, length: int = 24) -> str:
    payload = "\n".join([(p or "").strip().replace("\r\n", "\n") for p in parts])
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[: int(length)]


def _json_default(obj: Any):
    if isinstance(obj, Path):
        return str(obj)
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")


def write_json(path: Path, obj: Any, retries: int = 6, sleep_sec: float = 0.25) -> None:
    ensure_dir(path.parent)
    payload = json.dumps(obj, ensure_ascii=False, indent=2, default=_json_default) + "\n"
    last_error = None
    for attempt in range(max(1, int(retries))):
        tmp = path.with_suffix(path.suffix + f".{attempt}.tmp")
        try:
            tmp.write_text(payload, encoding="utf-8")
            tmp.replace(path)
            return
        except PermissionError as e:
            last_error = e
            time.sleep(float(sleep_sec) * float(attempt + 1))
        finally:
            try:
                if tmp.exists():
                    tmp.unlink()
            except Exception:
                pass
    if last_error is not None:
        raise last_error
    raise RuntimeError(f"Failed to write JSON: {path}")


def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def append_jsonl(path: Path, obj: Any) -> None:
    ensure_dir(path.parent)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False, default=_json_default) + "\n")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def inspect_pdf(path: Path) -> Dict[str, Any]:
    out: Dict[str, Any] = {
        "page_count": None,
        "has_outline": None,
        "inspect_status": "not_attempted",
    }
    if fitz is None:
        out["inspect_status"] = "fitz_unavailable"
        return out
    try:
        with fitz.open(path) as doc:
            out["page_count"] = int(doc.page_count)
            try:
                out["has_outline"] = bool(doc.get_toc())
            except Exception:
                out["has_outline"] = None
        out["inspect_status"] = "ok"
    except Exception as e:
        out["inspect_status"] = f"error:{type(e).__name__}"
    return out


print_section("Phase A.0 - Environment")
print_kv(
    {
        "repo_root": REPO_ROOT,
        "notebook_dir": NOTEBOOK_DIR,
        "openai_api_key_present": bool(OPENAI_API_KEY),
        "pymupdf_available": bool(fitz is not None),
        "utc_now": utc_now_iso(),
    }
)

Phase A.0 - Environment
repo_root                  <projektverzeichnis>
notebook_dir               <projektverzeichnis>\pdf-scan
openai_api_key_present     True
pymupdf_available          True
utc_now                    2026-03-13T12:00:34+00:00


In [3]:
# Phase A.1 - Config models, PDF source resolution, run artifacts, and logging helpers

@dataclass
class PdfSource:
    label: str
    path: Path


@dataclass
class RunArtifacts:
    config_json: Path
    pdf_manifest_json: Path
    query_plan_json: Path
    parser_dir: Path
    normalized_dir: Path
    retrieval_dir: Path
    rerank_dir: Path
    final_dir: Path
    logs_jsonl: Path
    run_log: Path
    metrics_json: Path

    @classmethod
    def from_run_dir(cls, run_dir: Path) -> "RunArtifacts":
        return cls(
            config_json=run_dir / "config.json",
            pdf_manifest_json=run_dir / "pdf_manifest.json",
            query_plan_json=run_dir / "query_plan.json",
            parser_dir=run_dir / "parser",
            normalized_dir=run_dir / "normalized",
            retrieval_dir=run_dir / "retrieval",
            rerank_dir=run_dir / "rerank",
            final_dir=run_dir / "final",
            logs_jsonl=run_dir / "logs.jsonl",
            run_log=run_dir / "run.log",
            metrics_json=run_dir / "metrics.json",
        )


@dataclass
class PipelineConfig:
    pipeline_version: str
    input_mode: str
    chapter_title: str
    chapter_spec_text: str
    runs_root: Path
    openai_api_key_present: bool
    force_rebuild_phase_a: bool
    pdf_sources: List[PdfSource]
    pdf_dir_raw: str
    pdf_glob: str
    pdf_recursive: bool
    max_pdfs: int
    benchmark_suite_manifest: str
    benchmark_suite_id: str
    benchmark_chapter_id: str

    def to_snapshot(self) -> Dict[str, Any]:
        return {
            "pipeline_version": self.pipeline_version,
            "input_mode": self.input_mode,
            "chapter_title": self.chapter_title,
            "chapter_spec_text_chars": len(self.chapter_spec_text),
            "runs_root": self.runs_root,
            "openai_api_key_present": self.openai_api_key_present,
            "force_rebuild_phase_a": self.force_rebuild_phase_a,
            "pdf_sources": [{"label": s.label, "path": str(s.path)} for s in self.pdf_sources],
            "pdf_dir_raw": self.pdf_dir_raw,
            "pdf_glob": self.pdf_glob,
            "pdf_recursive": self.pdf_recursive,
            "max_pdfs": self.max_pdfs,
            "benchmark_suite_manifest": self.benchmark_suite_manifest,
            "benchmark_suite_id": self.benchmark_suite_id,
            "benchmark_chapter_id": self.benchmark_chapter_id,
        }


@dataclass
class RunContext:
    repo_root: Path
    notebook_dir: Path
    run_id: str
    run_dir: Path
    artifacts: RunArtifacts

    def create_artifact_skeleton(self, overwrite: bool = False) -> None:
        ensure_dir(self.run_dir)
        ensure_dir(self.artifacts.parser_dir)
        ensure_dir(self.artifacts.normalized_dir)
        ensure_dir(self.artifacts.retrieval_dir)
        ensure_dir(self.artifacts.rerank_dir)
        ensure_dir(self.artifacts.final_dir)

        placeholders: Dict[Path, Any] = {
            self.artifacts.query_plan_json: {"status": "not_run", "phase": "query_planner"},
            self.artifacts.metrics_json: {"run_id": self.run_id, "stages": {}},
        }
        for path, payload in placeholders.items():
            if overwrite or (not path.exists()):
                write_json(path, payload)

        for path in [self.artifacts.logs_jsonl, self.artifacts.run_log]:
            if overwrite or (not path.exists()):
                ensure_dir(path.parent)
                path.write_text("", encoding="utf-8")

        for rel in [
            self.artifacts.normalized_dir / "documents.jsonl",
            self.artifacts.normalized_dir / "sections.jsonl",
            self.artifacts.normalized_dir / "passages.jsonl",
            self.artifacts.retrieval_dir / "fused_candidates.jsonl",
            self.artifacts.rerank_dir / "cross_encoder.jsonl",
            self.artifacts.final_dir / "output.json",
        ]:
            if overwrite or (not rel.exists()):
                ensure_dir(rel.parent)
                if rel.suffix == ".json":
                    write_json(rel, {"status": "not_run"})
                else:
                    rel.write_text("", encoding="utf-8")


def _resolve_existing_path(raw: str, *, expect_dir: bool) -> Path:
    p = Path(raw).expanduser()
    candidates = [p]
    if not p.is_absolute():
        candidates.extend([NOTEBOOK_DIR / p, REPO_ROOT / p, Path.cwd().resolve() / p])
    seen: List[Path] = []
    for cand in candidates:
        cand = cand.resolve()
        if cand in seen:
            continue
        seen.append(cand)
        if cand.exists() and ((cand.is_dir() and expect_dir) or (cand.is_file() and not expect_dir)):
            return cand
    return candidates[0].resolve()


def _normalize_pdf_sources(raw_sources: List[Dict[str, Any]]) -> List[PdfSource]:
    out: List[PdfSource] = []
    seen_labels: Dict[str, int] = {}
    for item in raw_sources or []:
        if not isinstance(item, dict):
            continue
        path_raw = str(item.get("path") or "").strip()
        if not path_raw:
            continue
        path = _resolve_existing_path(path_raw, expect_dir=False)
        if not path.exists():
            raise FileNotFoundError(f"PDF not found: {path}")
        label = str(item.get("label") or path.stem).strip() or path.stem
        n = seen_labels.get(label, 0) + 1
        seen_labels[label] = n
        if n > 1:
            label = f"{label} ({n})"
        out.append(PdfSource(label=label, path=path))
    return out


def resolve_pdf_sources() -> List[PdfSource]:
    explicit = _normalize_pdf_sources(PDF_SOURCES)
    if explicit:
        return explicit[: int(MAX_PDFS)]

    pdf_dir = str(PDF_DIR or "").strip()
    if not pdf_dir:
        raise RuntimeError("No PDFs configured. Set PDF_SOURCES or PDF_DIR.")

    root = _resolve_existing_path(pdf_dir, expect_dir=True)
    if not root.exists():
        raise FileNotFoundError(f"PDF_DIR not found: {root}")

    paths = sorted(root.rglob(PDF_GLOB) if bool(PDF_RECURSIVE) else root.glob(PDF_GLOB))
    paths = [p.resolve() for p in paths if p.is_file()]
    if not paths:
        raise FileNotFoundError(f"No PDFs found in {root} with pattern {PDF_GLOB!r}")

    out: List[PdfSource] = []
    seen_labels: Dict[str, int] = {}
    for path in paths[: int(MAX_PDFS)]:
        label = path.stem
        n = seen_labels.get(label, 0) + 1
        seen_labels[label] = n
        if n > 1:
            label = f"{label} ({n})"
        out.append(PdfSource(label=label, path=path))
    return out


def compute_run_id(chapter_title: str, chapter_spec_text: str, pipeline_version: str, manifest_rows: List[Dict[str, Any]]) -> str:
    doc_parts = [f"{row.get('label')}::{row.get('sha256')}" for row in manifest_rows]
    return stable_hash(pipeline_version, chapter_title, chapter_spec_text, "\n".join(doc_parts), length=24)


def load_metrics(run_ctx: RunContext) -> Dict[str, Any]:
    if run_ctx.artifacts.metrics_json.exists():
        try:
            return read_json(run_ctx.artifacts.metrics_json)
        except Exception:
            return {"run_id": run_ctx.run_id, "stages": {}}
    return {"run_id": run_ctx.run_id, "stages": {}}


def save_metrics(run_ctx: RunContext, metrics: Dict[str, Any]) -> None:
    write_json(run_ctx.artifacts.metrics_json, metrics)


def setup_run_logger(run_ctx: RunContext) -> logging.Logger:
    logger_name = f"pdf_scan_v2.{run_ctx.run_id}"
    logger = logging.getLogger(logger_name)
    logger.setLevel(logging.INFO)
    logger.handlers = []
    logger.propagate = False

    fh = logging.FileHandler(run_ctx.artifacts.run_log, encoding="utf-8")
    fh.setLevel(logging.INFO)
    fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(fh)
    return logger


def log_event(run_ctx: RunContext, *, stage: str, event: str, **payload: Any) -> None:
    append_jsonl(
        run_ctx.artifacts.logs_jsonl,
        {
            "ts_utc": utc_now_iso(),
            "stage": stage,
            "event": event,
            **payload,
        },
    )


@contextmanager
def stage_timer(run_ctx: RunContext, stage: str):
    t0 = time.perf_counter()
    try:
        yield
    except Exception as e:
        elapsed_ms = round((time.perf_counter() - t0) * 1000.0, 3)
        metrics = load_metrics(run_ctx)
        metrics.setdefault("stages", {}).setdefault(stage, {})["elapsed_ms"] = elapsed_ms
        metrics["stages"][stage]["failed_at_utc"] = utc_now_iso()
        metrics["stages"][stage]["last_error"] = {"type": type(e).__name__, "message": str(e)}
        save_metrics(run_ctx, metrics)
        logger = logging.getLogger(f"pdf_scan_v2.{run_ctx.run_id}")
        if getattr(logger, "handlers", None):
            logger.info("Stage failed | stage=%s | elapsed_ms=%s | error=%s: %s", stage, elapsed_ms, type(e).__name__, str(e))
        log_event(run_ctx, stage=stage, event="stage_failed", elapsed_ms=elapsed_ms, error_type=type(e).__name__, error_message=str(e))
        raise
    else:
        elapsed_ms = round((time.perf_counter() - t0) * 1000.0, 3)
        metrics = load_metrics(run_ctx)
        metrics.setdefault("stages", {}).setdefault(stage, {})["elapsed_ms"] = elapsed_ms
        metrics["stages"][stage]["finished_at_utc"] = utc_now_iso()
        save_metrics(run_ctx, metrics)
        logger = logging.getLogger(f"pdf_scan_v2.{run_ctx.run_id}")
        if getattr(logger, "handlers", None):
            logger.info("Stage finished | stage=%s | elapsed_ms=%s", stage, elapsed_ms)
        log_event(run_ctx, stage=stage, event="stage_finished", elapsed_ms=elapsed_ms)

In [4]:
# Phase A.2 - Resolve PDFs, create run context, write artifacts, and print QC summary

resolved_sources = resolve_pdf_sources()
if not resolved_sources:
    raise RuntimeError("resolve_pdf_sources() returned no PDFs.")

pdf_manifest_rows: List[Dict[str, Any]] = []
for src in resolved_sources:
    stat = src.path.stat()
    inspect = inspect_pdf(src.path)
    pdf_manifest_rows.append(
        {
            "label": src.label,
            "path": str(src.path),
            "file_name": src.path.name,
            "size_mb": round(float(stat.st_size) / (1024.0 * 1024.0), 3),
            "sha256": sha256_file(src.path),
            "page_count": inspect.get("page_count"),
            "has_outline": inspect.get("has_outline"),
            "inspect_status": inspect.get("inspect_status"),
            "mtime_utc": datetime.fromtimestamp(stat.st_mtime, tz=timezone.utc).replace(microsecond=0).isoformat(),
        }
    )

run_id = compute_run_id(CHAPTER_TITLE, CHAPTER_DESCRIPTION, PIPELINE_VERSION, pdf_manifest_rows)
runs_root = ensure_dir((NOTEBOOK_DIR / "runs").resolve())
run_dir = runs_root / run_id
artifacts = RunArtifacts.from_run_dir(run_dir)
run_ctx = RunContext(repo_root=REPO_ROOT, notebook_dir=NOTEBOOK_DIR, run_id=run_id, run_dir=run_dir, artifacts=artifacts)

cfg = PipelineConfig(
    pipeline_version=PIPELINE_VERSION,
    input_mode=INPUT_MODE,
    chapter_title=CHAPTER_TITLE,
    chapter_spec_text=CHAPTER_DESCRIPTION,
    runs_root=runs_root,
    openai_api_key_present=bool(OPENAI_API_KEY),
    force_rebuild_phase_a=bool(FORCE_REBUILD_PHASE_A),
    pdf_sources=resolved_sources,
    pdf_dir_raw=str(PDF_DIR or ""),
    pdf_glob=str(PDF_GLOB or "*.pdf"),
    pdf_recursive=bool(PDF_RECURSIVE),
    max_pdfs=int(MAX_PDFS),
    benchmark_suite_manifest=str(BENCHMARK_SUITE_MANIFEST_RESOLVED or ""),
    benchmark_suite_id=str(BENCHMARK_SUITE_ID or ""),
    benchmark_chapter_id=str(BENCHMARK_CHAPTER_ID or ""),
)

with stage_timer(run_ctx, "phase_a"):
    run_ctx.create_artifact_skeleton(overwrite=bool(FORCE_REBUILD_PHASE_A))
    logger = setup_run_logger(run_ctx)
    logger.info("Phase A initialized | run_id=%s | run_dir=%s", run_ctx.run_id, run_ctx.run_dir)

    write_json(run_ctx.artifacts.config_json, cfg.to_snapshot())
    write_json(
        run_ctx.artifacts.pdf_manifest_json,
        {
            "generated_at_utc": utc_now_iso(),
            "run_id": run_ctx.run_id,
            "pdf_count": len(pdf_manifest_rows),
            "pdfs": pdf_manifest_rows,
        },
    )

    metrics = load_metrics(run_ctx)
    metrics.setdefault("stages", {}).setdefault("phase_a", {}).update(
        {
            "initialized_at_utc": utc_now_iso(),
            "input_mode": INPUT_MODE,
            "pdf_count": len(pdf_manifest_rows),
            "has_openai_api_key": bool(OPENAI_API_KEY),
            "pymupdf_available": bool(fitz is not None),
            "benchmark_suite_id": BENCHMARK_SUITE_ID or "",
            "benchmark_chapter_id": BENCHMARK_CHAPTER_ID or "",
        }
    )
    save_metrics(run_ctx, metrics)

    log_event(
        run_ctx,
        stage="phase_a",
        event="run_initialized",
        run_id=run_ctx.run_id,
        run_dir=str(run_ctx.run_dir),
        input_mode=INPUT_MODE,
        pdf_count=len(pdf_manifest_rows),
        has_openai_api_key=bool(OPENAI_API_KEY),
        benchmark_suite_id=BENCHMARK_SUITE_ID or "",
        benchmark_chapter_id=BENCHMARK_CHAPTER_ID or "",
    )

expected_paths = [
    run_ctx.artifacts.config_json,
    run_ctx.artifacts.pdf_manifest_json,
    run_ctx.artifacts.query_plan_json,
    run_ctx.artifacts.parser_dir,
    run_ctx.artifacts.normalized_dir,
    run_ctx.artifacts.retrieval_dir,
    run_ctx.artifacts.rerank_dir,
    run_ctx.artifacts.final_dir,
    run_ctx.artifacts.logs_jsonl,
    run_ctx.artifacts.run_log,
    run_ctx.artifacts.metrics_json,
]
missing_paths = [str(p) for p in expected_paths if not p.exists()]

artifact_rows = [
    {"artifact": "config_json", "path": run_ctx.artifacts.config_json, "exists": run_ctx.artifacts.config_json.exists()},
    {"artifact": "pdf_manifest_json", "path": run_ctx.artifacts.pdf_manifest_json, "exists": run_ctx.artifacts.pdf_manifest_json.exists()},
    {"artifact": "query_plan_json", "path": run_ctx.artifacts.query_plan_json, "exists": run_ctx.artifacts.query_plan_json.exists()},
    {"artifact": "parser_dir", "path": run_ctx.artifacts.parser_dir, "exists": run_ctx.artifacts.parser_dir.exists()},
    {"artifact": "normalized_dir", "path": run_ctx.artifacts.normalized_dir, "exists": run_ctx.artifacts.normalized_dir.exists()},
    {"artifact": "retrieval_dir", "path": run_ctx.artifacts.retrieval_dir, "exists": run_ctx.artifacts.retrieval_dir.exists()},
    {"artifact": "rerank_dir", "path": run_ctx.artifacts.rerank_dir, "exists": run_ctx.artifacts.rerank_dir.exists()},
    {"artifact": "final_dir", "path": run_ctx.artifacts.final_dir, "exists": run_ctx.artifacts.final_dir.exists()},
    {"artifact": "logs_jsonl", "path": run_ctx.artifacts.logs_jsonl, "exists": run_ctx.artifacts.logs_jsonl.exists()},
    {"artifact": "metrics_json", "path": run_ctx.artifacts.metrics_json, "exists": run_ctx.artifacts.metrics_json.exists()},
]

qc_rows = []
qc_rows.append(
    qc_row(
        check="artifact_skeleton",
        status="OK" if not missing_paths else "FAIL",
        value="all present" if not missing_paths else ("missing: " + ", ".join(missing_paths[:4])),
        expected="all expected artifact paths exist",
        why="later phases rely on deterministic artifact locations",
        fix="re-run Phase A or inspect permissions / path resolution",
    )
)
qc_rows.append(
    qc_row(
        check="openai_api_key",
        status="OK" if bool(OPENAI_API_KEY) else "WARN",
        value=bool(OPENAI_API_KEY),
        expected="True before query-planning and LLM phases",
        why="later phases use the OpenAI API",
        fix="set OPENAI_API_KEY in .env before Phase D",
    )
)
qc_rows.append(
    qc_row(
        check="pymupdf_available",
        status="OK" if bool(fitz is not None) else "WARN",
        value=bool(fitz is not None),
        expected="True",
        why="PyMuPDF is used for PDF inspection and later fallback parsing",
        fix="install PyMuPDF before Phase B if this is False",
    )
)
qc_rows.append(
    qc_row(
        check="pdf_count",
        status="OK" if len(pdf_manifest_rows) >= 1 else "FAIL",
        value=len(pdf_manifest_rows),
        expected=">= 1",
        why="the pipeline needs at least one PDF input",
        fix="add PDFs to PDF_SOURCES or PDF_DIR",
    )
)

RUN_LOGGER = setup_run_logger(run_ctx)
RUN_CONTEXT = run_ctx
CONFIG = cfg
PDF_MANIFEST = pdf_manifest_rows

print_section("Phase A.2 - Run Context")
print_kv(
    {
        "run_id": run_ctx.run_id,
        "run_dir": run_ctx.run_dir,
        "input_mode": INPUT_MODE,
        "pdf_count": len(pdf_manifest_rows),
        "chapter_title": _truncate(CHAPTER_TITLE, 90),
        "pipeline_version": PIPELINE_VERSION,
        "benchmark_suite_id": BENCHMARK_SUITE_ID or "<none>",
        "benchmark_chapter_id": BENCHMARK_CHAPTER_ID or "<none>",
    }
)

print_section("Phase A.2 - PDF Manifest Preview")
print_table(
    pdf_manifest_rows,
    columns=["label", "file_name", "page_count", "has_outline", "size_mb", "inspect_status"],
    max_rows=20,
)

print_section("Phase A.2 - Artifact Preview")
print_table(artifact_rows, columns=["artifact", "exists", "path"], max_rows=20, max_col_width=70)

print_section("Phase A.2 - QC")
print_table(qc_rows, columns=["check", "status", "value", "expected", "why", "fix"], max_rows=20, max_col_width=46)

Phase A.2 - Run Context
run_id                     67507862e53171c27bfb2ac9
run_dir                    <projektverzeichnis>\pdf-scan\runs\67507862e53171c27bfb2ac9
input_mode                 small_gold
pdf_count                  5
chapter_title              Entscheidungspsychologie im Kontext unsicherer Kaufentscheidungen im Webshop-Kontext
pipeline_version           pdf_scan_v2
benchmark_suite_id         small_gold_webshop_decision_psychology_v1
benchmark_chapter_id       chapter_001_webshop_decision_psychology
Phase A.2 - PDF Manifest Preview
label                                                        | file_name                                                    | page_count | has_outline | size_mb | inspect_status
-------------------------------------------------------------+--------------------------------------------------------------+------------+-------------+---------+---------------
Consumers' Decision-Making Process on Social Commerce Pla... | Consumers’ Decision-Making Proc

---
# Phase B - Parser bundle, raw artifacts, and readability diagnostics
---


In [ ]:
# Phase B.0 - Parser bundle helpers and artifact writers

import importlib.metadata as importlib_metadata
import io
import json
import math
import re
import site
import sys
import time
import traceback
import warnings
from contextlib import redirect_stderr, redirect_stdout
from dataclasses import dataclass
from datetime import datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

OPTIONAL_IMPORT_ERRORS = {}

try:
    import fitz  # PyMuPDF
except Exception as e:
    fitz = None
    OPTIONAL_IMPORT_ERRORS["fitz"] = f"{type(e).__name__}: {e}"

try:
    from pypdf import PdfReader
except Exception as e:
    PdfReader = None
    OPTIONAL_IMPORT_ERRORS["pypdf"] = f"{type(e).__name__}: {e}"

try:
    from docling.document_converter import DocumentConverter
except Exception as e:
    DocumentConverter = None
    OPTIONAL_IMPORT_ERRORS["docling"] = f"{type(e).__name__}: {e}"

try:
    import requests
except Exception as e:
    requests = None
    OPTIONAL_IMPORT_ERRORS["requests"] = f"{type(e).__name__}: {e}"

try:
    from bs4 import BeautifulSoup
except Exception as e:
    BeautifulSoup = None
    OPTIONAL_IMPORT_ERRORS["bs4"] = f"{type(e).__name__}: {e}"


@dataclass
class PhaseBOptions:
    force_rebuild: bool = False
    doc_limit: Optional[int] = None
    include_doc_ids: Optional[List[str]] = None
    exclude_doc_ids: Optional[List[str]] = None
    min_page_words: int = 20
    min_doc_chars: int = 200
    try_docling: bool = True
    docling_page_limit: int = 200
    try_grobid: bool = True
    grobid_page_limit: int = 200
    grobid_base_url: str = ""
    grobid_process_path: str = "/api/processFulltextDocument"
    grobid_timeout_sec: int = 120
    grobid_consolidate_header: int = 0
    grobid_consolidate_citations: int = 0
    grobid_include_raw_citations: int = 0

    def normalized(self) -> "PhaseBOptions":
        return PhaseBOptions(
            force_rebuild=bool(self.force_rebuild),
            doc_limit=None if self.doc_limit is None else int(self.doc_limit),
            include_doc_ids=[str(x).strip() for x in (self.include_doc_ids or []) if str(x).strip()],
            exclude_doc_ids=[str(x).strip() for x in (self.exclude_doc_ids or []) if str(x).strip()],
            min_page_words=int(self.min_page_words),
            min_doc_chars=int(self.min_doc_chars),
            try_docling=bool(self.try_docling),
            docling_page_limit=int(self.docling_page_limit),
            try_grobid=bool(self.try_grobid),
            grobid_page_limit=int(self.grobid_page_limit),
            grobid_base_url=str(self.grobid_base_url or "").strip(),
            grobid_process_path=str(self.grobid_process_path or "/api/processFulltextDocument").strip() or "/api/processFulltextDocument",
            grobid_timeout_sec=int(self.grobid_timeout_sec),
            grobid_consolidate_header=int(self.grobid_consolidate_header),
            grobid_consolidate_citations=int(self.grobid_consolidate_citations),
            grobid_include_raw_citations=int(self.grobid_include_raw_citations),
        )


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def pkg_version(name: str) -> Optional[str]:
    try:
        return importlib_metadata.version(name)
    except Exception:
        return None


def json_safe(obj: Any) -> Any:
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, Enum):
        return obj.value
    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            return None
        return obj
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [json_safe(v) for v in obj]
    return obj


def write_json_atomic(path: Path, obj: Any, retries: int = 6, sleep_sec: float = 0.25) -> None:
    ensure_dir(path.parent)
    payload = json.dumps(json_safe(obj), ensure_ascii=False, indent=2) + "\n"
    last_error = None
    for attempt in range(max(1, int(retries))):
        tmp = path.with_suffix(path.suffix + f".{attempt}.tmp")
        try:
            tmp.write_text(payload, encoding="utf-8")
            tmp.replace(path)
            return
        except PermissionError as e:
            last_error = e
            time.sleep(float(sleep_sec) * float(attempt + 1))
        finally:
            try:
                if tmp.exists():
                    tmp.unlink()
            except Exception:
                pass
    if last_error is not None:
        raise last_error
    raise RuntimeError(f"Failed to write JSON atomically: {path}")


def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def write_jsonl_rows(path: Path, rows: List[Dict[str, Any]]) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(json_safe(row), ensure_ascii=False) + "\n")


def clean_text(text: Any) -> str:
    s = str(text or "")
    s = s.replace("\xad", "")
    s = s.replace("\u00a0", " ")
    s = s.replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()


def count_words(text: Any) -> int:
    return len(re.findall(r"\w+", str(text or ""), flags=re.UNICODE))


def slugify(text: str, max_len: int = 64) -> str:
    s = re.sub(r"[^A-Za-z0-9]+", "_", str(text or "").strip().lower()).strip("_")
    s = re.sub(r"_+", "_", s)
    return (s or "doc")[: int(max_len)]


def rel_to_run(run_dir: Path, path: Path) -> str:
    try:
        return str(path.relative_to(run_dir))
    except Exception:
        return str(path)


def short_blob(text: str, max_len: int = 12000) -> str:
    s = str(text or "")
    return s if len(s) <= max_len else (s[: max_len - 1] + "...")


def runtime_env_snapshot() -> Dict[str, Any]:
    try:
        user_site = site.getusersitepackages()
    except Exception:
        user_site = None
    try:
        site_packages = [str(p) for p in site.getsitepackages()]
    except Exception:
        site_packages = []
    return {
        "python_executable": sys.executable,
        "python_version": sys.version.split()[0],
        "python_prefix": sys.prefix,
        "python_base_prefix": getattr(sys, "base_prefix", sys.prefix),
        "cwd": str(Path.cwd()),
        "user_site": user_site,
        "site_packages": site_packages,
        "sys_path_preview": [str(x) for x in sys.path[:15]],
    }


def capture_python_noise(fn: Callable[[], Any]) -> Dict[str, Any]:
    stdout_buf = io.StringIO()
    stderr_buf = io.StringIO()
    with redirect_stdout(stdout_buf), redirect_stderr(stderr_buf), warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        result = fn()
    return {
        "result": result,
        "stdout": short_blob(stdout_buf.getvalue()),
        "stderr": short_blob(stderr_buf.getvalue()),
        "warnings": [short_blob(str(w.message), max_len=2000) for w in caught],
    }


def ping_grobid(base_url: str) -> Dict[str, Any]:
    out = {
        "configured": bool(base_url),
        "reachable": False,
        "status": "not_configured",
        "url": base_url,
        "error": None,
    }
    if not base_url:
        return out
    if requests is None:
        out["status"] = "requests_unavailable"
        return out
    try:
        resp = requests.get(base_url.rstrip("/") + "/api/isalive", timeout=10)
        out["reachable"] = bool(resp.ok)
        out["status"] = "alive" if resp.ok else f"http_{resp.status_code}"
    except Exception as e:
        out["status"] = f"error:{type(e).__name__}"
        out["error"] = str(e)
    return out


def detect_capabilities(grobid_base_url: str) -> Dict[str, Any]:
    return {
        "generated_at_utc": utc_now_iso(),
        "runtime": runtime_env_snapshot(),
        "fitz_available": bool(fitz is not None),
        "fitz_version": pkg_version("PyMuPDF"),
        "pypdf_available": bool(PdfReader is not None),
        "pypdf_version": pkg_version("pypdf"),
        "docling_available": bool(DocumentConverter is not None),
        "docling_version": pkg_version("docling"),
        "requests_available": bool(requests is not None),
        "requests_version": pkg_version("requests"),
        "bs4_available": bool(BeautifulSoup is not None),
        "bs4_version": pkg_version("beautifulsoup4"),
        "optional_import_errors": dict(OPTIONAL_IMPORT_ERRORS),
        "grobid": ping_grobid(grobid_base_url),
    }


def required_phase_b_kernel_packages(options: PhaseBOptions) -> List[str]:
    packages = ["PyMuPDF", "pypdf"]
    if bool(options.try_docling):
        packages.append("docling")
    if bool(options.try_grobid):
        packages.extend(["requests", "beautifulsoup4"])
    return packages


def missing_phase_b_kernel_packages(capabilities: Dict[str, Any], options: PhaseBOptions) -> List[str]:
    missing: List[str] = []
    if not bool(capabilities.get("fitz_available")):
        missing.append("PyMuPDF")
    if not bool(capabilities.get("pypdf_available")):
        missing.append("pypdf")
    if bool(options.try_docling) and not bool(capabilities.get("docling_available")):
        missing.append("docling")
    if bool(options.try_grobid) and not bool(capabilities.get("requests_available")):
        missing.append("requests")
    if bool(options.try_grobid) and not bool(capabilities.get("bs4_available")):
        missing.append("beautifulsoup4")
    return missing


def compute_doc_id(manifest_row: Dict[str, Any], stable_hash_fn: Optional[Callable[..., str]] = None) -> str:
    stem = slugify(Path(manifest_row.get("file_name") or "document.pdf").stem, max_len=48)
    digest = str(manifest_row.get("sha256") or "")[:12]
    if not digest and stable_hash_fn is not None:
        digest = stable_hash_fn(stem, str(manifest_row.get("path") or ""), length=12)
    digest = digest or "docbundle0000"
    return f"{stem}-{digest}"


def flatten_pypdf_outline(reader: Any, outline: Any) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []

    def walk(nodes: Any, level: int) -> None:
        for node in nodes or []:
            if isinstance(node, list):
                walk(node, level + 1)
                continue
            title = clean_text(getattr(node, "title", None) or str(node))
            page_num = None
            try:
                page_num = int(reader.get_destination_page_number(node)) + 1
            except Exception:
                page_num = None
            rows.append({"level": int(level), "title": title, "page": page_num})

    if isinstance(outline, list):
        walk(outline, 1)
    return rows


def extract_pypdf_bundle(path: Path) -> Dict[str, Any]:
    out: Dict[str, Any] = {
        "available": bool(PdfReader is not None),
        "status": "unavailable",
        "page_count": None,
        "metadata": {},
        "outline": [],
        "error": None,
    }
    if PdfReader is None:
        return out
    try:
        reader = PdfReader(str(path))
        out["page_count"] = int(len(reader.pages))
        out["metadata"] = {str(k): str(v) for k, v in dict(reader.metadata or {}).items()}
        out["outline"] = flatten_pypdf_outline(reader, getattr(reader, "outline", []))
        out["status"] = "ok"
    except Exception as e:
        out["status"] = f"error:{type(e).__name__}"
        out["error"] = str(e)
    return out


def normalize_fitz_block(block: Any, page_num: int, block_idx: int) -> Dict[str, Any]:
    row: Dict[str, Any] = {
        "page": int(page_num),
        "block_index": int(block_idx),
        "x0": None,
        "y0": None,
        "x1": None,
        "y1": None,
        "text": "",
        "block_no": None,
        "block_type": None,
        "char_len": 0,
        "word_count": 0,
    }
    if isinstance(block, (list, tuple)):
        vals = list(block)
        if len(vals) >= 4:
            row["x0"], row["y0"], row["x1"], row["y1"] = [float(v) if v is not None else None for v in vals[:4]]
        if len(vals) >= 5:
            row["text"] = clean_text(vals[4])
        if len(vals) >= 6:
            row["block_no"] = vals[5]
        if len(vals) >= 7:
            row["block_type"] = vals[6]
    row["char_len"] = len(row["text"])
    row["word_count"] = count_words(row["text"])
    return row


def extract_fitz_bundle(path: Path, min_page_words: int) -> Dict[str, Any]:
    out: Dict[str, Any] = {
        "available": bool(fitz is not None),
        "status": "unavailable",
        "page_count": None,
        "metadata": {},
        "outline": [],
        "pages": [],
        "blocks": [],
        "error": None,
    }
    if fitz is None:
        return out
    try:
        with fitz.open(path) as doc:
            out["page_count"] = int(doc.page_count)
            out["metadata"] = {str(k): str(v) for k, v in dict(doc.metadata or {}).items()}
            try:
                toc = doc.get_toc(simple=True) or []
            except Exception:
                toc = []
            out["outline"] = [
                {
                    "level": int(item[0]),
                    "title": clean_text(item[1]),
                    "page": int(item[2]) if len(item) > 2 and item[2] is not None else None,
                }
                for item in toc
            ]
            for page_index in range(doc.page_count):
                page = doc[page_index]
                try:
                    page_text = clean_text(page.get_text("text", sort=True))
                except TypeError:
                    page_text = clean_text(page.get_text("text"))
                page_word_count = count_words(page_text)
                try:
                    raw_blocks = page.get_text("blocks", sort=True)
                except TypeError:
                    raw_blocks = page.get_text("blocks")

                block_rows = []
                for block_idx, block in enumerate(raw_blocks or []):
                    row = normalize_fitz_block(block, page_index + 1, block_idx)
                    if row["text"]:
                        block_rows.append(row)
                out["blocks"].extend(block_rows)
                out["pages"].append(
                    {
                        "page": int(page_index + 1),
                        "text": page_text,
                        "char_len": len(page_text),
                        "word_count": page_word_count,
                        "has_text": bool(page_word_count > 0),
                        "has_substantive_text": bool(page_word_count >= int(min_page_words)),
                    }
                )
        out["status"] = "ok"
    except Exception as e:
        out["status"] = f"error:{type(e).__name__}"
        out["error"] = str(e)
    return out


_DOCLING_CONVERTER = None


def get_docling_converter() -> Any:
    global _DOCLING_CONVERTER
    if DocumentConverter is None:
        return None
    if _DOCLING_CONVERTER is None:
        _DOCLING_CONVERTER = DocumentConverter()
    return _DOCLING_CONVERTER


def extract_docling_bundle(path: Path, page_count: Optional[int], options: PhaseBOptions) -> Dict[str, Any]:
    out: Dict[str, Any] = {
        "available": bool(DocumentConverter is not None),
        "enabled": False,
        "status": "unavailable",
        "error": None,
        "stdout": "",
        "stderr": "",
        "warnings": [],
        "result": None,
        "document": None,
        "markdown_preview": None,
    }
    if DocumentConverter is None:
        return out
    if not bool(options.try_docling):
        out["status"] = "disabled"
        return out
    if page_count and int(page_count) > int(options.docling_page_limit):
        out["status"] = "skipped_page_limit"
        out["error"] = f"page_count={page_count} exceeds docling_page_limit={options.docling_page_limit}"
        return out

    out["enabled"] = True
    try:
        captured = capture_python_noise(
            lambda: get_docling_converter().convert(
                path,
                raises_on_error=False,
                max_num_pages=int(options.docling_page_limit),
            )
        )
        res = captured["result"]
        out["stdout"] = captured["stdout"]
        out["stderr"] = captured["stderr"]
        out["warnings"] = captured["warnings"]
        raw_dump = json_safe(res.model_dump()) if hasattr(res, "model_dump") else None
        out["result"] = {
            "status": raw_dump.get("status") if isinstance(raw_dump, dict) else None,
            "errors": raw_dump.get("errors") if isinstance(raw_dump, dict) else None,
            "input": raw_dump.get("input") if isinstance(raw_dump, dict) else None,
            "timings": raw_dump.get("timings") if isinstance(raw_dump, dict) else None,
            "confidence": raw_dump.get("confidence") if isinstance(raw_dump, dict) else None,
        }
        out["status"] = str(out["result"].get("status") or "unknown")
        if out["status"] == "success":
            doc = getattr(res, "document", None)
            if doc is not None:
                out["document"] = json_safe(doc.export_to_dict())
                try:
                    out["markdown_preview"] = short_blob(doc.export_to_markdown(), max_len=8000)
                except Exception:
                    out["markdown_preview"] = None
    except Exception as e:
        out["status"] = f"error:{type(e).__name__}"
        out["error"] = str(e)
        out["stderr"] = short_blob(traceback.format_exc())
    return out


def should_try_grobid(
    manifest_row: Dict[str, Any],
    page_count: Optional[int],
    options: PhaseBOptions,
    capabilities: Dict[str, Any],
) -> tuple[bool, str]:
    if not bool(options.try_grobid):
        return False, "disabled"
    grobid = capabilities.get("grobid", {})
    if not bool(grobid.get("configured")):
        return False, "not_configured"
    if not bool(grobid.get("reachable")):
        return False, f"service_{grobid.get('status') or 'unreachable'}"
    if page_count and int(page_count) > int(options.grobid_page_limit):
        return False, "skipped_page_limit"
    return True, "pdf_under_page_limit"


def summarize_grobid_xml(xml_text: str) -> Dict[str, Any]:
    xml_text = str(xml_text or "")
    if not xml_text:
        return {"status": "empty"}
    if BeautifulSoup is None:
        return {"status": "bs4_unavailable"}
    try:
        soup = BeautifulSoup(xml_text, "xml")
        head_texts = []
        for tag in soup.find_all("head"):
            txt = clean_text(tag.get_text(" ", strip=True))
            if txt:
                head_texts.append(txt)
        title_tag = soup.find("title")
        return {
            "status": "ok",
            "title": clean_text(title_tag.get_text(" ", strip=True)) if title_tag else None,
            "section_head_count": len(head_texts),
            "section_head_preview": head_texts[:15],
            "has_abstract": bool(soup.find("abstract")),
            "has_bibliography": bool(soup.find("listBibl")),
        }
    except Exception as e:
        return {"status": f"error:{type(e).__name__}", "error": str(e)}


def extract_grobid_bundle(
    path: Path,
    manifest_row: Dict[str, Any],
    page_count: Optional[int],
    options: PhaseBOptions,
    capabilities: Dict[str, Any],
) -> Dict[str, Any]:
    enabled, reason = should_try_grobid(manifest_row, page_count, options, capabilities)
    out: Dict[str, Any] = {
        "available": bool(requests is not None),
        "enabled": bool(enabled),
        "status": "not_attempted",
        "reason": reason,
        "error": None,
        "xml_text": None,
        "summary": None,
    }
    if not enabled:
        out["status"] = reason
        return out
    try:
        with path.open("rb") as f:
            response = requests.post(
                options.grobid_base_url.rstrip("/") + options.grobid_process_path,
                files={"input": (path.name, f, "application/pdf")},
                data={
                    "consolidateHeader": str(int(options.grobid_consolidate_header)),
                    "consolidateCitations": str(int(options.grobid_consolidate_citations)),
                    "includeRawCitations": str(int(options.grobid_include_raw_citations)),
                },
                timeout=int(options.grobid_timeout_sec),
            )
        if not response.ok:
            out["status"] = f"http_{response.status_code}"
            out["error"] = short_blob(response.text, max_len=4000)
            return out
        xml_text = response.text or ""
        out["xml_text"] = xml_text
        out["summary"] = summarize_grobid_xml(xml_text)
        out["status"] = "ok" if xml_text.strip() else "empty_response"
    except Exception as e:
        out["status"] = f"error:{type(e).__name__}"
        out["error"] = str(e)
    return out


def _docling_success_like(status: Any) -> bool:
    return str(status or "") in {"success", "partial_success"}


def compute_phase_b_counts(
    summary_rows: List[Dict[str, Any]],
    capabilities: Dict[str, Any],
    selected_count: int,
) -> Dict[str, Any]:
    low_coverage_docs = [
        row["doc_id"]
        for row in summary_rows
        if row.get("pages_with_text_pct") is None or float(row.get("pages_with_text_pct") or 0.0) < 50.0
    ]
    unreadable_docs = [row["doc_id"] for row in summary_rows if not row.get("readable_without_ocr")]
    cached_docs = [row["doc_id"] for row in summary_rows if row.get("cached")]
    runtime_capability_mismatch_docs = [row["doc_id"] for row in summary_rows if row.get("runtime_capability_mismatch")]
    fallback_docs = [row["doc_id"] for row in summary_rows if row.get("fallback_activated")]
    docling_success_like_docs = [row["doc_id"] for row in summary_rows if _docling_success_like(row.get("docling_status"))]
    docling_success_docs = [row["doc_id"] for row in summary_rows if str(row.get("docling_status") or "") == "success"]
    docling_partial_docs = [row["doc_id"] for row in summary_rows if str(row.get("docling_status") or "") == "partial_success"]
    grobid_ok_docs = [row["doc_id"] for row in summary_rows if str(row.get("grobid_status") or "") == "ok"]
    outline_docs = [row["doc_id"] for row in summary_rows if int(row.get("outline_count") or 0) > 0]

    return {
        "selected_count": int(selected_count),
        "documents_processed": len(summary_rows),
        "fitz_available": bool(capabilities.get("fitz_available")),
        "pypdf_available": bool(capabilities.get("pypdf_available")),
        "docling_available": bool(capabilities.get("docling_available")),
        "grobid_configured": bool(capabilities.get("grobid", {}).get("configured")),
        "grobid_reachable": bool(capabilities.get("grobid", {}).get("reachable")),
        "readable_without_ocr_count": sum(1 for row in summary_rows if row.get("readable_without_ocr")),
        "unreadable_without_ocr_count": len(unreadable_docs),
        "unreadable_without_ocr_docs": unreadable_docs,
        "low_text_coverage_count": len(low_coverage_docs),
        "low_text_coverage_docs": low_coverage_docs,
        "cached_doc_count": len(cached_docs),
        "cached_doc_ids": cached_docs,
        "runtime_capability_mismatch_count": len(runtime_capability_mismatch_docs),
        "runtime_capability_mismatch_docs": runtime_capability_mismatch_docs,
        "fallback_activated_count": len(fallback_docs),
        "fallback_activated_docs": fallback_docs,
        "docling_success_like_count": len(docling_success_like_docs),
        "docling_success_count": len(docling_success_docs),
        "docling_partial_success_count": len(docling_partial_docs),
        "grobid_success_count": len(grobid_ok_docs),
        "outline_doc_count": len(outline_docs),
    }


def build_phase_b_assessment(
    summary_rows: List[Dict[str, Any]],
    capabilities: Dict[str, Any],
    selected_count: int,
) -> Dict[str, Any]:
    counts = compute_phase_b_counts(summary_rows, capabilities, selected_count)
    failures: List[str] = []
    warnings_list: List[str] = []
    infos: List[str] = []
    next_actions: List[str] = []

    if not counts["fitz_available"]:
        failures.append("PyMuPDF is unavailable. Phase B cannot satisfy the deterministic fallback contract.")
        next_actions.append("Install PyMuPDF in the notebook kernel and rerun Phase B.")
    if counts["documents_processed"] != counts["selected_count"]:
        failures.append(
            f"Only {counts['documents_processed']} of {counts['selected_count']} selected PDFs produced parser bundles."
        )
        next_actions.append("Inspect parser logs and diagnostics for missing bundle outputs.")
    if counts["unreadable_without_ocr_count"] > 0:
        failures.append(
            f"{counts['unreadable_without_ocr_count']} PDF(s) were not readable without OCR in a digital-only benchmark."
        )
        next_actions.append("Inspect the unreadable PDFs and confirm they are digitally extractable.")

    if not counts["pypdf_available"]:
        warnings_list.append("pypdf is unavailable in the current notebook runtime, so independent outline validation is missing.")
        next_actions.append("Install pypdf in the notebook kernel and inspect phase_b_runtime.json if the kernel path is unclear.")
    if not counts["docling_available"]:
        warnings_list.append("Docling is unavailable in the current notebook runtime.")
        next_actions.append("Install docling in the notebook kernel and inspect phase_b_runtime.json if the kernel path is unclear.")
    if counts["docling_success_like_count"] < max(1, counts["documents_processed"] // 2):
        warnings_list.append(
            "Docling succeeded only on a minority of documents, so Phase C will lean heavily on fallback structure signals."
        )
        next_actions.append("Review docling.json diagnostics and consider adjusting page limits or runtime setup.")
    if counts["low_text_coverage_count"] > 0:
        warnings_list.append(
            f"{counts['low_text_coverage_count']} PDF(s) had low extracted text coverage and may carry weak section evidence."
        )
        next_actions.append("Inspect pymupdf_pages.jsonl for low-coverage documents before Phase C.")
    if counts["fallback_activated_count"] > 0:
        warnings_list.append(
            f"Fallback parsing was activated for {counts['fallback_activated_count']} document(s)."
        )
    if counts["runtime_capability_mismatch_count"] > 0:
        warnings_list.append(
            "Some cached parser bundles were created under a different runtime capability profile than the current notebook session."
        )
        next_actions.append("Set force_rebuild=True for Phase B if you need a clean run under the current environment.")
    if not counts["grobid_configured"]:
        warnings_list.append("GROBID is not configured, so the scholarly enhancement lane is absent.")
        next_actions.append("Configure GROBID_URL or GROBID_BASE_URL if you want TEI structure recovery.")
    elif not counts["grobid_reachable"]:
        warnings_list.append("GROBID is configured but not reachable.")
        next_actions.append("Start or fix the GROBID service before rerunning Phase B.")

    if counts["outline_doc_count"] > 0:
        infos.append(f"{counts['outline_doc_count']} PDF(s) expose outline/bookmark structure already.")
    if counts["readable_without_ocr_count"] == counts["documents_processed"] and counts["documents_processed"] > 0:
        infos.append("All processed PDFs were readable without OCR.")

    if failures:
        status = "fail"
        quality_band = "degraded"
        can_continue = False
    elif warnings_list:
        status = "success_with_warnings"
        quality_band = "acceptable_with_issues"
        can_continue = True
    else:
        status = "success"
        quality_band = "strong"
        can_continue = True

    warnings_list = list(dict.fromkeys(warnings_list))
    next_actions = list(dict.fromkeys(next_actions))
    infos = list(dict.fromkeys(infos))

    return {
        "generated_at_utc": utc_now_iso(),
        "phase": "phase_b",
        "status": status,
        "quality_band": quality_band,
        "can_continue_to_next_phase": bool(can_continue),
        "failures": failures,
        "warnings": warnings_list,
        "info": infos,
        "recommended_next_actions": next_actions,
        "counts": counts,
    }


def build_qc_rows(
    summary_rows: List[Dict[str, Any]],
    capabilities: Dict[str, Any],
    selected_count: int,
    assessment: Optional[Dict[str, Any]] = None,
) -> List[Dict[str, Any]]:
    def qc_row(check: str, status: str, value: Any, expected: str, why: str, fix: str) -> Dict[str, Any]:
        return {
            "check": str(check),
            "status": str(status),
            "value": str(value),
            "expected": str(expected),
            "why": str(why),
            "fix": str(fix),
        }

    counts = (assessment or build_phase_b_assessment(summary_rows, capabilities, selected_count)).get("counts", {})
    low_coverage_docs = list(counts.get("low_text_coverage_docs") or [])
    unreadable_docs = list(counts.get("unreadable_without_ocr_docs") or [])
    docling_success_count = int(counts.get("docling_success_count") or 0)

    qc_rows = []
    qc_rows.append(
        qc_row(
            "documents_processed",
            "OK" if int(counts.get("documents_processed") or 0) == int(selected_count) else "FAIL",
            counts.get("documents_processed"),
            str(selected_count),
            "every selected PDF should emit a parser bundle",
            "inspect diagnostics.json for any missing document output",
        )
    )
    qc_rows.append(
        qc_row(
            "fitz_available",
            "OK" if bool(capabilities.get("fitz_available")) else "FAIL",
            bool(capabilities.get("fitz_available")),
            "True",
            "PyMuPDF is the deterministic fallback and text-coverage lane",
            "install PyMuPDF before rerunning Phase B",
        )
    )
    qc_rows.append(
        qc_row(
            "pypdf_available",
            "OK" if bool(capabilities.get("pypdf_available")) else "WARN",
            bool(capabilities.get("pypdf_available")),
            "True",
            "pypdf provides an independent metadata and outline lane",
            "install pypdf before rerunning Phase B",
        )
    )
    qc_rows.append(
        qc_row(
            "unreadable_without_ocr",
            "OK" if not unreadable_docs else "WARN",
            "none" if not unreadable_docs else ", ".join(unreadable_docs[:4]),
            "none",
            "this benchmark is restricted to digital PDFs with extractable text",
            "inspect diagnostics for any document flagged as unreadable",
        )
    )
    qc_rows.append(
        qc_row(
            "low_text_coverage_docs",
            "OK" if not low_coverage_docs else "WARN",
            "none" if not low_coverage_docs else ", ".join(low_coverage_docs[:4]),
            "none below 50% page text coverage",
            "low coverage usually indicates parser trouble or image-heavy pages",
            "inspect pymupdf_pages.jsonl and diagnostics for the affected PDFs",
        )
    )
    qc_rows.append(
        qc_row(
            "cache_runtime_mismatch",
            "OK" if int(counts.get("runtime_capability_mismatch_count") or 0) == 0 else "WARN",
            counts.get("runtime_capability_mismatch_count"),
            "0",
            "cached results from a different runtime can make the phase look healthier than the current kernel supports",
            "set force_rebuild=True for a clean run under the current environment",
        )
    )
    qc_rows.append(
        qc_row(
            "docling_success_count",
            "OK" if docling_success_count >= 1 else "WARN",
            docling_success_count,
            ">= 1 on a healthy environment, otherwise fallback path must carry",
            "Docling is the preferred structure-aware parser when it works",
            "inspect docling.json stdout/stderr and consider tuning page limits or environment setup",
        )
    )
    qc_rows.append(
        qc_row(
            "grobid_service",
            "OK" if bool(capabilities.get("grobid", {}).get("reachable")) else "WARN",
            capabilities.get("grobid", {}).get("status"),
            "alive when scholarly enhancement is configured",
            "GROBID is optional but valuable for article/report structure recovery",
            "start a GROBID service and set GROBID_URL or GROBID_BASE_URL",
        )
    )
    if assessment is not None:
        qc_rows.append(
            qc_row(
                "phase_b_status",
                "OK" if assessment.get("status") == "success" else ("WARN" if assessment.get("status") == "success_with_warnings" else "FAIL"),
                assessment.get("status"),
                "success or success_with_warnings",
                "this is the persisted phase-level verdict used to judge whether the stage is healthy enough to continue",
                "inspect phase_b_assessment.json and the document diagnostics before continuing",
            )
        )
    return qc_rows


def run_phase_b(
    run_ctx: Any,
    pdf_manifest: List[Dict[str, Any]],
    options: PhaseBOptions,
    *,
    stable_hash_fn: Optional[Callable[..., str]] = None,
    log_event_fn: Optional[Callable[..., Any]] = None,
    run_logger: Optional[Any] = None,
) -> Dict[str, Any]:
    options = options.normalized()
    capabilities = detect_capabilities(options.grobid_base_url)
    required_kernel_packages = required_phase_b_kernel_packages(options)
    missing_kernel_packages = missing_phase_b_kernel_packages(capabilities, options)

    parser_dir = ensure_dir(Path(run_ctx.artifacts.parser_dir))
    config_path = parser_dir / "phase_b_config.json"
    runtime_path = parser_dir / "phase_b_runtime.json"
    summary_path = parser_dir / "phase_b_summary.json"
    assessment_path = parser_dir / "phase_b_assessment.json"
    index_path = parser_dir / "parsed_document_bundles.jsonl"

    write_json_atomic(
        runtime_path,
        {
            "generated_at_utc": utc_now_iso(),
            "phase": "phase_b",
            "options": json_safe(options.__dict__),
            "required_kernel_packages": required_kernel_packages,
            "missing_kernel_packages": missing_kernel_packages,
            "capabilities": capabilities,
        },
    )

    config_payload = {
        "generated_at_utc": utc_now_iso(),
        "phase": "phase_b",
        "options": json_safe(options.__dict__),
        "capabilities": capabilities,
        "runtime_path": rel_to_run(Path(run_ctx.run_dir), runtime_path),
    }
    write_json_atomic(config_path, config_payload)

    selected_rows: List[Dict[str, Any]] = []
    include_doc_ids = set(options.include_doc_ids or [])
    exclude_doc_ids = set(options.exclude_doc_ids or [])

    for manifest_row in list(pdf_manifest or []):
        doc_id = compute_doc_id(manifest_row, stable_hash_fn=stable_hash_fn)
        if include_doc_ids and doc_id not in include_doc_ids:
            continue
        if doc_id in exclude_doc_ids:
            continue
        row = dict(manifest_row)
        row["doc_id"] = doc_id
        selected_rows.append(row)

    if options.doc_limit is not None:
        selected_rows = selected_rows[: int(options.doc_limit)]
    if not selected_rows:
        raise RuntimeError("Phase B selected zero PDFs after filtering. Adjust PhaseBOptions filters.")
    if run_logger is not None:
        run_logger.info(
            "Phase B runtime | python=%s | missing_kernel_packages=%s",
            capabilities.get("runtime", {}).get("python_executable"),
            ",".join(missing_kernel_packages) if missing_kernel_packages else "none",
        )
        run_logger.info(
            "Phase B started | selected=%s | force_rebuild=%s | fitz=%s | pypdf=%s | docling=%s | grobid=%s",
            len(selected_rows),
            options.force_rebuild,
            capabilities.get("fitz_available"),
            capabilities.get("pypdf_available"),
            capabilities.get("docling_available"),
            capabilities.get("grobid", {}).get("status"),
        )

    summary_rows: List[Dict[str, Any]] = []
    bundle_rows: List[Dict[str, Any]] = []

    for manifest_row in selected_rows:
        doc_id = str(manifest_row["doc_id"])
        source_path = Path(str(manifest_row["path"])).resolve()
        doc_dir = ensure_dir(parser_dir / doc_id)
        metadata_path = doc_dir / "metadata.json"
        diagnostics_path = doc_dir / "diagnostics.json"
        fitz_pages_path = doc_dir / "pymupdf_pages.jsonl"
        fitz_blocks_path = doc_dir / "pymupdf_blocks.jsonl"
        docling_path = doc_dir / "docling.json"
        grobid_summary_path = doc_dir / "grobid_summary.json"
        grobid_xml_path = doc_dir / "grobid.tei.xml"

        required_cache_paths = [metadata_path, diagnostics_path, fitz_pages_path, fitz_blocks_path, docling_path, grobid_summary_path]
        if (not options.force_rebuild) and all(p.exists() for p in required_cache_paths):
            cached_diag = read_json(diagnostics_path)
            cached_summary = dict(cached_diag.get("summary_row") or {})
            if cached_summary:
                runtime_snapshot = dict(cached_diag.get("runtime_capabilities_snapshot") or {})
                cached_options_snapshot = dict(cached_diag.get("phase_b_options_snapshot") or {})
                parser_statuses = dict(cached_diag.get("parser_statuses") or {})
                mismatch_fields: List[str] = []
                for field in ["fitz_available", "pypdf_available", "docling_available"]:
                    current_val = bool(capabilities.get(field))
                    cached_val = runtime_snapshot.get(field)
                    if isinstance(cached_val, bool):
                        if current_val != cached_val:
                            mismatch_fields.append(field)
                        continue
                    if field == "fitz_available":
                        cached_status = str(parser_statuses.get("fitz") or "")
                        if current_val and cached_status in {"", "unavailable"}:
                            mismatch_fields.append(field)
                        if (not current_val) and cached_status not in {"", "unavailable"}:
                            mismatch_fields.append(field)
                    if field == "pypdf_available":
                        cached_status = str(parser_statuses.get("pypdf") or "")
                        if current_val and cached_status in {"", "unavailable"}:
                            mismatch_fields.append(field)
                        if (not current_val) and cached_status not in {"", "unavailable"}:
                            mismatch_fields.append(field)
                    if field == "docling_available":
                        cached_status = str(parser_statuses.get("docling") or "")
                        docling_was_enabled = bool(cached_options_snapshot.get("try_docling", True))
                        if docling_was_enabled and current_val and cached_status in {"", "unavailable", "disabled"}:
                            mismatch_fields.append(field)
                        if (not current_val) and cached_status not in {"", "unavailable", "disabled"}:
                            mismatch_fields.append(field)
                cached_summary["cached"] = True
                cached_summary["cached_from_generated_at_utc"] = cached_diag.get("generated_at_utc")
                cached_summary["runtime_capability_mismatch"] = bool(mismatch_fields)
                cached_summary["runtime_capability_mismatch_fields"] = mismatch_fields
                summary_rows.append(cached_summary)
                bundle_rows.append(dict(cached_diag.get("bundle_row") or {}))
                if run_logger is not None:
                    run_logger.info(
                        "Phase B cached document | doc_id=%s | bundle_status=%s | docling=%s | grobid=%s | mismatch=%s",
                        doc_id,
                        cached_diag.get("bundle_status"),
                        cached_diag.get("parser_statuses", {}).get("docling"),
                        cached_diag.get("parser_statuses", {}).get("grobid"),
                        bool(mismatch_fields),
                    )
                if log_event_fn is not None:
                    log_event_fn(
                        run_ctx,
                        stage="phase_b",
                        event="document_reused_from_cache",
                        doc_id=doc_id,
                        bundle_status=cached_diag.get("bundle_status"),
                        docling_status=cached_diag.get("parser_statuses", {}).get("docling"),
                        grobid_status=cached_diag.get("parser_statuses", {}).get("grobid"),
                        runtime_capability_mismatch=bool(mismatch_fields),
                    )
                continue

        t0 = time.perf_counter()
        fitz_bundle = extract_fitz_bundle(source_path, min_page_words=options.min_page_words)
        pypdf_bundle = extract_pypdf_bundle(source_path)
        page_count = fitz_bundle.get("page_count") or pypdf_bundle.get("page_count") or manifest_row.get("page_count")
        fitz_pages = list(fitz_bundle.get("pages") or [])
        fitz_blocks = list(fitz_bundle.get("blocks") or [])
        pages_with_text = sum(1 for row in fitz_pages if row.get("has_text"))
        pages_with_substantive_text = sum(1 for row in fitz_pages if row.get("has_substantive_text"))
        total_chars = sum(int(row.get("char_len") or 0) for row in fitz_pages)
        pct_pages_with_text = round((pages_with_text / float(page_count)) * 100.0, 2) if page_count else None
        pct_pages_with_substantive_text = round((pages_with_substantive_text / float(page_count)) * 100.0, 2) if page_count else None
        readable_without_ocr = bool(total_chars >= int(options.min_doc_chars) and pages_with_substantive_text >= 1)
        ocr_required_or_unreadable = not readable_without_ocr
        outline_count = max(len(fitz_bundle.get("outline") or []), len(pypdf_bundle.get("outline") or []))
        page_count_agrees = (
            fitz_bundle.get("page_count") is None
            or pypdf_bundle.get("page_count") is None
            or int(fitz_bundle.get("page_count")) == int(pypdf_bundle.get("page_count"))
        )

        docling_bundle = extract_docling_bundle(source_path, page_count, options)
        docling_success = str(docling_bundle.get("status") or "") == "success"
        grobid_bundle = extract_grobid_bundle(source_path, manifest_row, page_count, options, capabilities)
        fallback_activated = bool(readable_without_ocr and not docling_success and str(fitz_bundle.get("status") or "") == "ok")
        bundle_status = "ok" if readable_without_ocr and str(fitz_bundle.get("status") or "") == "ok" else "needs_attention"
        elapsed_ms = round((time.perf_counter() - t0) * 1000.0, 3)

        metadata_payload = {
            "generated_at_utc": utc_now_iso(),
            "phase": "phase_b",
            "doc_id": doc_id,
            "label": manifest_row.get("label"),
            "source_path": str(source_path),
            "file_name": source_path.name,
            "sha256": manifest_row.get("sha256"),
            "size_mb": manifest_row.get("size_mb"),
            "page_count": page_count,
            "page_count_sources": {
                "phase_a_manifest": manifest_row.get("page_count"),
                "fitz": fitz_bundle.get("page_count"),
                "pypdf": pypdf_bundle.get("page_count"),
                "agree": bool(page_count_agrees),
            },
            "outline_counts": {
                "fitz": len(fitz_bundle.get("outline") or []),
                "pypdf": len(pypdf_bundle.get("outline") or []),
            },
            "text_coverage": {
                "pages_with_text": pages_with_text,
                "pages_with_substantive_text": pages_with_substantive_text,
                "percent_pages_with_text": pct_pages_with_text,
                "percent_pages_with_substantive_text": pct_pages_with_substantive_text,
                "total_chars": total_chars,
                "readable_without_ocr": bool(readable_without_ocr),
                "ocr_required_or_unreadable": bool(ocr_required_or_unreadable),
            },
            "fitz": {
                "status": fitz_bundle.get("status"),
                "metadata": fitz_bundle.get("metadata"),
                "outline": fitz_bundle.get("outline"),
                "error": fitz_bundle.get("error"),
            },
            "pypdf": {
                "status": pypdf_bundle.get("status"),
                "metadata": pypdf_bundle.get("metadata"),
                "outline": pypdf_bundle.get("outline"),
                "error": pypdf_bundle.get("error"),
            },
        }

        summary_row = {
            "doc_id": doc_id,
            "file_name": source_path.name,
            "page_count": page_count,
            "outline_count": outline_count,
            "pages_with_text_pct": pct_pages_with_text,
            "substantive_text_pct": pct_pages_with_substantive_text,
            "readable_without_ocr": bool(readable_without_ocr),
            "docling_status": docling_bundle.get("status"),
            "grobid_status": grobid_bundle.get("status"),
            "fallback_activated": bool(fallback_activated),
            "page_count_agrees": bool(page_count_agrees),
            "elapsed_ms": elapsed_ms,
            "cached": False,
            "cached_from_generated_at_utc": None,
            "runtime_capability_mismatch": False,
            "runtime_capability_mismatch_fields": [],
        }

        bundle_row = {
            "doc_id": doc_id,
            "source_path": str(source_path),
            "metadata_json": rel_to_run(Path(run_ctx.run_dir), metadata_path),
            "diagnostics_json": rel_to_run(Path(run_ctx.run_dir), diagnostics_path),
            "pymupdf_pages_jsonl": rel_to_run(Path(run_ctx.run_dir), fitz_pages_path),
            "pymupdf_blocks_jsonl": rel_to_run(Path(run_ctx.run_dir), fitz_blocks_path),
            "docling_json": rel_to_run(Path(run_ctx.run_dir), docling_path),
            "grobid_summary_json": rel_to_run(Path(run_ctx.run_dir), grobid_summary_path),
            "grobid_tei_xml": rel_to_run(Path(run_ctx.run_dir), grobid_xml_path) if grobid_bundle.get("xml_text") else None,
            "bundle_status": bundle_status,
        }

        diagnostics_payload = {
            "generated_at_utc": utc_now_iso(),
            "phase": "phase_b",
            "doc_id": doc_id,
            "bundle_status": bundle_status,
            "summary_row": summary_row,
            "bundle_row": bundle_row,
            "readable_without_ocr": bool(readable_without_ocr),
            "ocr_required_or_unreadable": bool(ocr_required_or_unreadable),
            "fallback_activated": bool(fallback_activated),
            "page_count_agrees": bool(page_count_agrees),
            "parser_statuses": {
                "fitz": fitz_bundle.get("status"),
                "pypdf": pypdf_bundle.get("status"),
                "docling": docling_bundle.get("status"),
                "grobid": grobid_bundle.get("status"),
            },
            "runtime_capabilities_snapshot": {
                "fitz_available": bool(capabilities.get("fitz_available")),
                "pypdf_available": bool(capabilities.get("pypdf_available")),
                "docling_available": bool(capabilities.get("docling_available")),
                "grobid_configured": bool(capabilities.get("grobid", {}).get("configured")),
                "grobid_reachable": bool(capabilities.get("grobid", {}).get("reachable")),
            },
            "phase_b_options_snapshot": json_safe(options.__dict__),
            "artifact_paths": bundle_row,
        }

        write_json_atomic(metadata_path, metadata_payload)
        write_jsonl_rows(fitz_pages_path, fitz_pages)
        write_jsonl_rows(fitz_blocks_path, fitz_blocks)
        write_json_atomic(docling_path, docling_bundle)
        write_json_atomic(grobid_summary_path, {k: v for k, v in grobid_bundle.items() if k != "xml_text"})
        if grobid_bundle.get("xml_text"):
            grobid_xml_path.write_text(str(grobid_bundle["xml_text"]), encoding="utf-8")
        elif grobid_xml_path.exists() and options.force_rebuild:
            grobid_xml_path.unlink()
        write_json_atomic(diagnostics_path, diagnostics_payload)

        summary_rows.append(summary_row)
        bundle_rows.append(bundle_row)

        if run_logger is not None:
            run_logger.info(
                "Phase B parsed document | doc_id=%s | page_count=%s | fitz=%s | pypdf=%s | docling=%s | grobid=%s | fallback=%s | elapsed_ms=%s",
                doc_id,
                page_count,
                fitz_bundle.get("status"),
                pypdf_bundle.get("status"),
                docling_bundle.get("status"),
                grobid_bundle.get("status"),
                bool(fallback_activated),
                elapsed_ms,
            )
        if log_event_fn is not None:
            log_event_fn(
                run_ctx,
                stage="phase_b",
                event="document_parsed",
                doc_id=doc_id,
                source_path=str(source_path),
                page_count=page_count,
                fitz_status=fitz_bundle.get("status"),
                pypdf_status=pypdf_bundle.get("status"),
                docling_status=docling_bundle.get("status"),
                grobid_status=grobid_bundle.get("status"),
                readable_without_ocr=bool(readable_without_ocr),
                fallback_activated=bool(fallback_activated),
                elapsed_ms=elapsed_ms,
            )

    assessment = build_phase_b_assessment(summary_rows, capabilities, len(selected_rows))
    qc_rows = build_qc_rows(summary_rows, capabilities, len(selected_rows), assessment=assessment)

    write_json_atomic(
        summary_path,
        {
            "generated_at_utc": utc_now_iso(),
            "run_id": run_ctx.run_id,
            "phase": "phase_b",
            "options": json_safe(options.__dict__),
            "runtime_path": rel_to_run(Path(run_ctx.run_dir), runtime_path),
            "capabilities": capabilities,
            "assessment": assessment,
            "qc_rows": qc_rows,
            "documents": summary_rows,
            "artifacts": bundle_rows,
        },
    )
    write_json_atomic(
        assessment_path,
        {
            "generated_at_utc": utc_now_iso(),
            "run_id": run_ctx.run_id,
            "phase": "phase_b",
            "assessment": assessment,
            "qc_rows": qc_rows,
            "runtime_path": rel_to_run(Path(run_ctx.run_dir), runtime_path),
            "summary_path": rel_to_run(Path(run_ctx.run_dir), summary_path),
            "index_path": rel_to_run(Path(run_ctx.run_dir), index_path),
        },
    )
    write_jsonl_rows(index_path, bundle_rows)
    if run_logger is not None:
        run_logger.info(
            "Phase B completed | status=%s | quality=%s | processed=%s | cached=%s | fallback=%s | warnings=%s | failures=%s",
            assessment.get("status"),
            assessment.get("quality_band"),
            assessment.get("counts", {}).get("documents_processed"),
            assessment.get("counts", {}).get("cached_doc_count"),
            assessment.get("counts", {}).get("fallback_activated_count"),
            len(assessment.get("warnings") or []),
            len(assessment.get("failures") or []),
        )

    metrics_update = {
        "initialized_at_utc": utc_now_iso(),
        "document_count": len(summary_rows),
        "readable_document_count": sum(1 for row in summary_rows if row.get("readable_without_ocr")),
        "docling_success_count": sum(1 for row in summary_rows if str(row.get("docling_status") or "") == "success"),
        "docling_success_like_count": sum(1 for row in summary_rows if _docling_success_like(row.get("docling_status"))),
        "grobid_success_count": sum(1 for row in summary_rows if str(row.get("grobid_status") or "") == "ok"),
        "grobid_reachable": bool(capabilities.get("grobid", {}).get("reachable")),
        "cached_doc_count": sum(1 for row in summary_rows if row.get("cached")),
        "runtime_capability_mismatch_count": sum(1 for row in summary_rows if row.get("runtime_capability_mismatch")),
        "status": assessment.get("status"),
        "quality_band": assessment.get("quality_band"),
        "can_continue_to_next_phase": assessment.get("can_continue_to_next_phase"),
        "warning_count": len(assessment.get("warnings") or []),
        "failure_count": len(assessment.get("failures") or []),
        "assessment_path": rel_to_run(Path(run_ctx.run_dir), assessment_path),
    }

    return {
        "config_path": config_path,
        "runtime_path": runtime_path,
        "summary_path": summary_path,
        "assessment_path": assessment_path,
        "index_path": index_path,
        "capabilities": capabilities,
        "required_kernel_packages": required_kernel_packages,
        "missing_kernel_packages": missing_kernel_packages,
        "summary_rows": summary_rows,
        "bundle_rows": bundle_rows,
        "metrics_update": metrics_update,
        "assessment": assessment,
        "qc_rows": qc_rows,
        "selected_count": len(selected_rows),
    }


In [ ]:
# Phase B.1 - Run parser bundles and print QC summary

phase_b_options = PhaseBOptions(
    force_rebuild=True,
    doc_limit=None,
    include_doc_ids=[],
    exclude_doc_ids=[],
    min_page_words=20,
    min_doc_chars=200,
    try_docling=True,
    docling_page_limit=200,
    try_grobid=True,
    grobid_page_limit=200,
    grobid_base_url=(os.getenv("GROBID_URL") or os.getenv("GROBID_BASE_URL") or "").strip(),
    grobid_process_path="/api/processFulltextDocument",
    grobid_timeout_sec=120,
    grobid_consolidate_header=0,
    grobid_consolidate_citations=0,
    grobid_include_raw_citations=0,
)

phase_b_logger = setup_run_logger(RUN_CONTEXT)

with stage_timer(RUN_CONTEXT, "phase_b"):
    phase_b_result = run_phase_b(
        RUN_CONTEXT,
        PDF_MANIFEST,
        phase_b_options,
        stable_hash_fn=stable_hash,
        log_event_fn=log_event,
        run_logger=phase_b_logger,
    )
    metrics = load_metrics(RUN_CONTEXT)
    metrics.setdefault("stages", {}).setdefault("phase_b", {}).update(phase_b_result["metrics_update"])
    save_metrics(RUN_CONTEXT, metrics)

phase_b_rel = lambda path: rel_to_run(Path(RUN_CONTEXT.run_dir), Path(path))

PHASE_B_RESULT = phase_b_result
PHASE_B_SUMMARY = phase_b_result["summary_rows"]
PHASE_B_BUNDLES = phase_b_result["bundle_rows"]

docling_success_count = sum(1 for row in PHASE_B_SUMMARY if str(row.get("docling_status") or "") == "success")
grobid_success_count = sum(1 for row in PHASE_B_SUMMARY if str(row.get("grobid_status") or "") == "ok")
fallback_activated_docs = sum(1 for row in PHASE_B_SUMMARY if row.get("fallback_activated"))

print_section("Phase B - Parser Capabilities")
print_kv(
    {
        "fitz_available": phase_b_result["capabilities"].get("fitz_available"),
        "pypdf_available": phase_b_result["capabilities"].get("pypdf_available"),
        "docling_available": phase_b_result["capabilities"].get("docling_available"),
        "python_executable": phase_b_result["capabilities"].get("runtime", {}).get("python_executable"),
        "grobid_status": phase_b_result["capabilities"].get("grobid", {}).get("status"),
        "selected_documents": phase_b_result["selected_count"],
        "docling_success_count": docling_success_count,
        "grobid_success_count": grobid_success_count,
    }
)

print_section("Phase B - What Happened")
print_kv(
    {
        "phase_b_config_json": phase_b_rel(phase_b_result["config_path"]),
        "phase_b_runtime_json": phase_b_rel(phase_b_result["runtime_path"]),
        "phase_b_summary_json": phase_b_rel(phase_b_result["summary_path"]),
        "phase_b_assessment_json": phase_b_rel(phase_b_result["assessment_path"]),
        "parsed_document_bundles_jsonl": phase_b_rel(phase_b_result["index_path"]),
        "documents_processed": len(PHASE_B_SUMMARY),
        "readable_without_ocr": sum(1 for row in PHASE_B_SUMMARY if row.get("readable_without_ocr")),
        "fallback_activated_docs": fallback_activated_docs,
        "missing_kernel_packages": ", ".join(phase_b_result.get("missing_kernel_packages") or []) or "none",
        "phase_status": phase_b_result["assessment"].get("status"),
        "quality_band": phase_b_result["assessment"].get("quality_band"),
    }
)

print_section("Phase B - Document Summary")
print_table(
    PHASE_B_SUMMARY,
    columns=[
        "doc_id",
        "file_name",
        "page_count",
        "outline_count",
        "pages_with_text_pct",
        "docling_status",
        "grobid_status",
        "fallback_activated",
    ],
    max_rows=20,
    max_col_width=44,
)

print_section("Phase B - Artifact Preview")
print_table(
    PHASE_B_BUNDLES,
    columns=["doc_id", "metadata_json", "pymupdf_blocks_jsonl", "docling_json", "grobid_tei_xml", "bundle_status"],
    max_rows=20,
    max_col_width=44,
)

print_section("Phase B - Assessment")
print_kv(
    {
        "status": phase_b_result["assessment"].get("status"),
        "quality_band": phase_b_result["assessment"].get("quality_band"),
        "can_continue": phase_b_result["assessment"].get("can_continue_to_next_phase"),
        "warning_count": len(phase_b_result["assessment"].get("warnings") or []),
        "failure_count": len(phase_b_result["assessment"].get("failures") or []),
        "cached_doc_count": phase_b_result["assessment"].get("counts", {}).get("cached_doc_count"),
        "runtime_capability_mismatch_count": phase_b_result["assessment"].get("counts", {}).get("runtime_capability_mismatch_count"),
    }
)

print_section("Phase B - QC")
print_table(
    phase_b_result["qc_rows"],
    columns=["check", "status", "value", "expected", "why", "fix"],
    max_rows=20,
    max_col_width=46,
)
